# tether, shape A
One process, one agent, the games the gateway lists, in its order. The code and the learned lineage are the datasets', copied and checked by sha256; the agent carries on from that lineage.


In [ ]:
import os
import sys
import time

T0 = time.time()
# the competition: a notebook runs at most 9 hours, internet off (the competition page, read 2026-10-30)
KAGGLE_WALL_S = 9 * 3600
# PLAY's whole-run margin (play_chain.sh: e4b_run.py ... 300); its end path measured 104-119 s (PLAY24 rounds 1-6)
MARGIN_S = 300
# rule A's slice (Isaiah 2026-10-23), at the values Isaiah set for Kaggle A 2026-10-30 21:28 ("like 300s-400 seconds")
SLICE_MIN_S, SLICE_MAX_S = 300, 400
# the learned lineage is required: this agent is born once and carries on from it (Isaiah 2026-10-30 21:28)
REQUIRE_LINEAGE = True
INPUT = os.environ.get("NB_INPUT", "/kaggle/input")
WORKING = os.environ.get("NB_WORKING", "/kaggle/working")
GATEWAY = os.environ.get("NB_GATEWAY", "http://gateway:8001")
WHEELS = os.path.join(INPUT, "competitions", "arc-prize-2026-arc-agi-3", "arc_agi_3_wheels")
N_GAMES = os.environ.get("NB_N_GAMES")      # local runs only; on Kaggle every listed game is played
RERUN = bool(os.getenv("KAGGLE_IS_COMPETITION_RERUN"))
DEADLINE_AT = T0 + KAGGLE_WALL_S - MARGIN_S
print(f"T0 {T0:.1f} deadline_at {DEADLINE_AT:.1f} rerun {RERUN} slice {SLICE_MIN_S}/{SLICE_MAX_S} "
      f"input {INPUT} working {WORKING} gateway {GATEWAY} n_games {N_GAMES}")


In [ ]:
import platform
import shutil

mem = {}
if os.path.isfile("/proc/meminfo"):
    with open("/proc/meminfo") as fh:
        mem = {ln.split(":")[0]: ln.split(":")[1].strip() for ln in fh if ":" in ln}
cg = None
for p in ("/sys/fs/cgroup/memory.max", "/sys/fs/cgroup/memory/memory.limit_in_bytes"):
    if os.path.isfile(p):
        with open(p) as fh:
            cg = (p, fh.read().strip())
        break
print(f"python {sys.version.split()[0]} {platform.platform()}")
print(f"nproc os.cpu_count {os.cpu_count()} sched_getaffinity "
      f"{len(os.sched_getaffinity(0)) if hasattr(os, 'sched_getaffinity') else None}")
print(f"MemTotal {mem.get('MemTotal')} MemAvailable {mem.get('MemAvailable')} cgroup {cg}")
print(f"disk working free {shutil.disk_usage(WORKING if os.path.isdir(WORKING) else '.').free}")


In [ ]:
import importlib.metadata as md
import subprocess

if os.path.isdir(WHEELS):
    r = subprocess.run([sys.executable, "-m", "pip", "install", "--no-index", "--find-links", WHEELS,
                        "arc-agi", "python-dotenv"], capture_output=True, text=True)
    print(r.stdout[-2000:], r.stderr[-2000:])
    assert r.returncode == 0, "the offline install failed"
else:
    print(f"no wheelhouse at {WHEELS}: the interpreter's own packages are used")
for p in ("arc-agi", "arcengine", "numpy", "requests", "pydantic"):
    try:
        print(p, md.version(p))
    except md.PackageNotFoundError:
        print(p, "NOT INSTALLED")


In [ ]:
import glob
import hashlib
import json

MARK = "TETHER_KAGGLE_DATASET.json"
found = sorted(set(glob.glob(os.path.join(INPUT, "*", MARK))
                   + glob.glob(os.path.join(INPUT, "datasets", "*", "*", MARK))))
assert len(found) == 1, ("the dataset marker, under /kaggle/input/<slug> or /kaggle/input/datasets/<owner>/<slug>",
                         found)
DS = os.path.dirname(found[0])
with open(found[0]) as fh:
    META = json.load(fh)
print("dataset", DS, json.dumps(META))
CODE = os.path.join(WORKING, "tether")          # a writable copy: the input mount is read-only
assert not os.path.exists(CODE), ("a copy already exists", CODE)
shutil.copytree(os.path.join(DS, "code"), CODE)
with open(os.path.join(DS, "MANIFEST.sha256")) as fh:
    rows = [ln.rstrip("\n").split("  ", 1) for ln in fh if ln.strip()]


def _sha(rel):
    with open(os.path.join(CODE, rel), "rb") as fh:
        return hashlib.sha256(fh.read()).hexdigest()


bad = [rel for h, rel in rows if _sha(rel) != h]
assert len(rows) == META["files"] and not bad, ("the copy differs from the manifest", len(rows), bad[:5])
print(f"copied {len(rows)} files to {CODE}; every sha256 equals the manifest")
r = subprocess.run([sys.executable, "-B", "-c", "import tether, arc_holdout; print('agent imports ok')"],
                   cwd=CODE, capture_output=True, text=True, env={**os.environ, "PYTHONDONTWRITEBYTECODE": "1"})
print(r.stdout[-1000:], r.stderr[-2000:])
assert r.returncode == 0, "the agent does not import on this interpreter"


In [ ]:
LMARK = "TETHER_KAGGLE_LINEAGE.json"
lfound = sorted(set(glob.glob(os.path.join(INPUT, "*", LMARK))
                    + glob.glob(os.path.join(INPUT, "datasets", "*", "*", LMARK))))
assert len(lfound) <= 1, ("more than one lineage dataset", lfound)
assert lfound or not REQUIRE_LINEAGE, "no lineage dataset attached, and the run requires one"
LIN = os.path.join(WORKING, "lineage")          # a writable copy: the agent writes its generations into it
assert not os.path.exists(LIN), ("a lineage copy already exists", LIN)
if lfound:
    LDS = os.path.dirname(lfound[0])
    with open(lfound[0]) as fh:
        LMETA = json.load(fh)
    print("lineage dataset", LDS, json.dumps(LMETA))
    t = time.time()
    shutil.copytree(os.path.join(LDS, "lineage"), LIN)
    with open(os.path.join(LDS, "MANIFEST.sha256")) as fh:
        lrows = [ln.rstrip("\n").split("  ", 1) for ln in fh if ln.strip()]

    def _lsha(rel):
        h = hashlib.sha256()
        with open(os.path.join(LIN, rel), "rb") as fh:
            for b in iter(lambda: fh.read(1 << 20), b""):
                h.update(b)
        return h.hexdigest()

    lbad = [rel for h, rel in lrows if _lsha(rel) != h]
    assert len(lrows) == LMETA["names"] and not lbad, ("the lineage copy differs", len(lrows), lbad[:5])
    r = subprocess.run([sys.executable, "-B", "-c", "import sys, lineage; print(lineage.head(sys.argv[1])[0])", LIN],
                       cwd=CODE, capture_output=True, text=True, env={**os.environ, "PYTHONDONTWRITEBYTECODE": "1"})
    assert r.returncode == 0 and r.stdout.strip() == LMETA["head"], ("the copy's head", r.stdout, r.stderr[-1000:])
    print(f"lineage copied and checked in {time.time() - t:.1f} s: {len(lrows)} names, head {LMETA['head']}")
else:
    print("no lineage dataset: the agent starts from the library alone")


In [ ]:
import urllib.request

OUT = os.path.join(WORKING, "run_a")
if RERUN:
    t = time.time()
    while True:                                 # the gateway, as the competition's sample waits for it (600 s)
        try:
            n = len(json.loads(urllib.request.urlopen(GATEWAY + "/api/games", timeout=10).read()))
            break
        except Exception as e:
            assert time.time() - t < 600, ("the gateway never answered", repr(e))
            time.sleep(5)
    print(f"gateway up after {time.time() - t:.1f} s, listing {n} games")
    NO_ENVS = os.path.join(WORKING, "no_local_envs")   # empty: the Arcade's list is the gateway's alone
    os.makedirs(NO_ENVS, exist_ok=True)
    env = {k: v for k, v in os.environ.items() if not k.startswith("TETHER_")}
    # as the competition's sample sets them; the key is the sample's own placeholder, never ours
    env.update(OPERATION_MODE="online", ARC_BASE_URL=GATEWAY + "/", ARC_API_KEY="test-key-123",
               ENVIRONMENTS_DIR=NO_ENVS, RECORDINGS_DIR=os.path.join(WORKING, "recordings"),
               ONLY_RESET_LEVELS="true", PYTHONHASHSEED="0", PYTHONDONTWRITEBYTECODE="1", MPLBACKEND="agg")
    cmd = [sys.executable, "-B", os.path.join(CODE, "kaggle", "run_a.py"), CODE, OUT, LIN,
           str(SLICE_MIN_S), str(SLICE_MAX_S), repr(DEADLINE_AT)] + ([N_GAMES] if N_GAMES else [])
    print("run", cmd)
    p = subprocess.run(cmd, env=env, cwd=CODE, stdin=subprocess.DEVNULL)
    print(f"run_a rc {p.returncode} wall {time.time() - T0:.1f} s since T0")
    for f in ("switch.tsv", "stream.result.json", "kaggle_a.json", "stream.traceback.txt"):
        f = os.path.join(OUT, f)
        if os.path.isfile(f):
            with open(f, encoding="utf-8") as fh:
                print("=====", f)
                print(fh.read()[-6000:])


In [ ]:
# Non-rerun mode: produce a dummy submission (the competition's sample, its import moved under the test: a rerun
# never needs pandas)
if not os.getenv('KAGGLE_IS_COMPETITION_RERUN'):
    import pandas as pd

    submission = pd.DataFrame(
        data=[['1_0', '1', True, 1]],
        columns=['row_id', 'game_id', 'end_of_game', 'score'])
    submission.to_parquet(os.path.join(WORKING, 'submission.parquet'), index=False)
